In [ ]:
import nltk, warnings
warnings.filterwarnings('ignore')
for pkg in ['punkt_tab', 'stopwords', 'wordnet', 'movie_reviews']:
  nltk.download(pkg, quiet = True)
print('NLP environment ready!')

NLP environment ready!


In [ ]:
from nltk.corpus import movie_reviews
docs, labels = [], []
for cat in movie_reviews.categories():
  for fid in movie_reviews.fileids(cat):
    docs.append(movie_reviews.raw(fid))
    labels.append(1 if cat == 'pos' else 0)
print('Documents:', len(docs), '| positive:', sum(labels), '| negative:', len(labels) - sum(labels))
print('First review(first 60 chars):', repr(docs[0][:60]))

Documents: 2000 | positive: 1000 | negative: 1000
First review(first 60 chars): 'plot : two teen couples go to a church party , drink and the'


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import numpy as np
tf = TfidfVectorizer(max_features = 5000, ngram_range = (1,2), stop_words = 'english')
X = tf.fit_transform(docs)
Xtr, Xte, ytr, yte = train_test_split(X, labels, test_size = 0.2, random_state = 42, stratify = labels)
clf = LogisticRegression(max_iter = 1000).fit(Xtr, ytr)
print('Accuracy:', round(clf.score(Xte, yte), 3))
names, coef = tf.get_feature_names_out(), clf.coef_[0]
print('Most positive:', [names[i] for i in np.argsort(coef)[-6:]][::-1])
print('Most negative:', [names[i] for i in np.argsort(coef)[:6]])
tests = ['A brilliant, moving film with superb acting.',
         'Boring  plot and a complete waste of time.',
         'The movie was not bad at all.']
print('Predictions:', list(clf.predict(tf.transform(tests))))

Accuracy: 0.823
Most positive: ['great', 'life', 'excellent', 'war', 'family', 'perfect']
Most negative: ['bad', 'worst', 'plot', 'boring', 'movie', 'supposed']
Predictions: [np.int64(1), np.int64(0), np.int64(0)]


In [ ]:
import spacy
from collections import Counter
nlp = spacy.load('en_core_web_sm')
article_news = ('Sundar Pichai announched on Monday that Google will invest '
                '$2 billion in India by 2027. The plan was discussed in New Delhi '
                'with officials from the Ministry of Electronics, and Reliance '
                'Industries confirmed a partnership last week.')
doc = nlp(article_news)
for ent in doc.ents:
  print(f' {ent.text:<28} {ent.label_}')
print('Entity counts:', dict(Counter(e.label_ for e in doc.ents)))
from spacy import displacy
displacy.render(doc,style='ent', jupyter=True)

 Sundar Pichai                PERSON
 Monday                       DATE
 Google                       ORG
 $2 billion                   MONEY
 India                        GPE
 2027                         DATE
 New Delhi                    GPE
 the Ministry of Electronics  ORG
 Reliance Industries          ORG
 last week                    DATE
Entity counts: {'PERSON': 1, 'DATE': 3, 'ORG': 3, 'MONEY': 1, 'GPE': 2}


In [ ]:
import re
text4 = ('Priya Sharma completed B.Sc. Data Science at KRMU at 2024. '
         'Contact her at priya.sharma@example.com or +91 9876543210'
         'before 15/08/2026 for the machine learning internship position.')
d4 = nlp(text4)
print('Noun chunks:', [c.text for c in d4.noun_chunks])
emails = re.findall(r'[\w.+-]+@[\w-]+\.[\w.]+', text4)
dates = re.findall(r'\d{1,2}[/-]\d{1,2}[\-]\d{2,4}|\b\d{4}\b', text4)
phones = re.findall(r'(?:\+91][- ]?)?\d{10}', text4)
print('Structured record:', {'emails' : emails, 'dates' : dates, 'phones' : phones})


Noun chunks: ['Priya Sharma', 'B.Sc', 'Data Science', 'KRMU', 'her', 'the machine', 'internship position']
Structured record: {'emails': ['priya.sharma@example.com'], 'dates': ['2024', '2026'], 'phones': ['9876543210']}
